# Perfis com identificadores sintéticos para o agente
Execute depois da Silver. Cria uma cópia Delta sem modificar a origem.
Cada linha representa uma proposta e recebe uma identidade fictícia independente.
Não é possível reconstruir pessoas ou vínculos entre propostas sem uma chave na origem.


In [0]:
from pyspark.sql import functions as F, Window

ORIGEM = 'workspace.default.silver_propostas_credito'
DESTINO = 'workspace.default.silver_perfis_credito_demo'

if ORIGEM == DESTINO:
    raise ValueError('Origem e destino precisam ser diferentes.')
if spark.catalog.tableExists(DESTINO):
    raise ValueError(f'{DESTINO} já existe. Reutilize a tabela para manter os identificadores estáveis.')

base = spark.table(ORIGEM)
obrigatorias = {'nota_credito_score', 'historico_inadimplencia_previo', 'renda_anual'}
faltantes = obrigatorias - set(base.columns)
if faltantes:
    raise ValueError(f'Adapte o mapeamento da Silver. Colunas ausentes: {sorted(faltantes)}')
novas = {'id_proposta_demo', 'cpf', 'identificador_sintetico'}
if novas & set(base.columns):
    raise ValueError(f'A origem já contém colunas reservadas: {sorted(novas & set(base.columns))}. Confira o esquema antes de continuar.')
quantidade = base.count()
if not 0 < quantidade <= 1_000_000:
    raise ValueError('Este notebook de demonstração espera de 1 a 1 milhão de registros.')


In [0]:
# Sequência global adequada ao pequeno dataset de portfólio.
# A associação entre linha e número não é reproduzível após recriar a tabela.
# Por isso os números são materializados uma única vez e a execução não sobrescreve o destino.
perfis = base.withColumn('id_proposta_demo', F.row_number().over(Window.orderBy(F.monotonically_increasing_id())))
base_cpf = F.lpad(F.col('id_proposta_demo').cast('string'), 9, '0')
soma = sum(F.substring(base_cpf, pos, 1).cast('int') * (11 - pos) for pos in range(1, 10))
resto = soma % 11
primeiro_digito_valido = F.when(resto < 2, F.lit(0)).otherwise(11 - resto)
# Altera deliberadamente o primeiro dígito verificador: nenhum CPF é válido.
primeiro_digito_invalido = ((primeiro_digito_valido + 1) % 10).cast('string')
perfis = (perfis.withColumn('cpf', F.concat(base_cpf, primeiro_digito_invalido, F.lit('0')))
          .withColumn('identificador_sintetico', F.lit(True)))

perfis.createOrReplaceTempView("perfis_demo_preparados")

spark.sql("""
    CREATE TABLE workspace.default.silver_perfis_credito_demo
    USING DELTA
    AS SELECT * FROM perfis_demo_preparados
""")



In [0]:
# Leia do destino persistido: esses são os identificadores para testar no terminal.
resultado = spark.table(DESTINO)
resumo = resultado.agg(F.count('*').alias('linhas'), F.countDistinct('cpf').alias('cpfs_unicos'),
                       F.countDistinct('id_proposta_demo').alias('ids_unicos')).first()
if not (resumo.linhas == resumo.cpfs_unicos == resumo.ids_unicos):
    raise ValueError('A validação de unicidade falhou. Não use o destino antes de investigar.')
print(f'Tabela de demonstração criada: {DESTINO} | {resumo.linhas} perfis sintéticos')
display(resultado.select('id_proposta_demo', 'cpf', 'nota_credito_score',
                         'historico_inadimplencia_previo', 'renda_anual', 'identificador_sintetico')
        .orderBy('id_proposta_demo').limit(10))
